# DAT535 Project – Group 24: Norgespris and electricity use
## Notebook 01: Scope, Dataset and Bronze layer

This notebook implements the first three sections of our *Data Engineering Project Definition Guide (v3)*:

| Section | What this notebook does |
|---|---|
| 1. Project statement | Writes down the scope we build against |
| 2. Dataset | Describes the sources, then **profiles the real files with Spark** (section 4 of this notebook) |
| 3. Bronze layer | Downloads / copies raw files **unchanged**, writes a manifest with checksums, skips duplicates, flags schema changes |

**How this notebook fits the repo**

* The same ingestion code is in `bronze_pipeline.py` (generated from the code cells in sections 3.1–3.9), so GitHub Actions can
  run it without Jupyter: `python run_pipeline.py bronze`.
* Dev and prod are separated by two environment variables, set by the workflows in `.github/workflows/`:
  `NORGESPRIS_ENV` (`dev`/`prod`) and `NORGESPRIS_OUTPUT_DIR` (`~/norgespris-data/dev` or `~/norgespris-data/prod`).
  When you run the notebook yourself it defaults to **dev**.
* Credentials (Frost) are read from the file **`.env`** in the repo root – never written in the notebook, never committed to git.
* Tests for the Bronze code: `pytest -q tests/`.

**Section 5 lists problems we found in the definition and in the data – read it before starting Silver.**

---
# 1. Project statement (scope)

**What we build:** a batch pipeline in PySpark with Bronze, Silver and Gold layers that reads data from
Elhub (electricity), MET/Frost (temperature) and SSB (population).

**Question it answers:** *Has daily electricity use per inhabitant in each municipality changed after Norgespris
started (1 October 2025), when we take temperature into account – and in which municipalities is the change largest?*

**Result:** clean Silver tables, a Gold table at grain *municipality × day*, and a simple dashboard with
municipal time series and a before/after comparison.

**Users:** our group and the examiner; possibly analysts interested in the same question.

**Out of scope (on purpose):** single households (EV charging, heat pumps, wood stoves), electricity prices,
real-time/streaming processing.

**Why this scope:** only open data, a clear before/after date, and real Spark work
(hourly → daily aggregation of ~15 million rows per year, joins on municipality and date) that still fits our machines.

**Analysis period:** from **November 2023** (the oldest data Elhub's API still serves – see section 5) to the last complete month.
That gives two winters before Norgespris (2023/24 and 2024/25) and at least one after.

---
# 2. Dataset (source and suitability)

| # | Dataset (folder name in Bronze) | From | Format | One row / record is | How we get it |
|---|---|---|---|---|---|
| 1 | `consumption_per_group_municipality_hour` | Elhub Energy Data API | ZIP with CSV | kWh and number of metering points per **municipality × hour × consumption group** | API CSV export: last ~3 years in one file |
| 2 | `norgespris_consumption_per_group_eac_mba` | Elhub Energy Data API | ZIP with CSV | kWh **with / without** Norgespris per **day × price area × group × annual-consumption band** | API CSV export |
| 3 | `norgespris_count_per_municipality_consumption_group` | Elhub open data (data.elhub.no) | CSV (gzip) | Number of metering points **with Norgespris** per municipality and group | Download, full history (*optional*) |
| 4 | `population_07459` | SSB table 07459 | CSV `;` | Population 1 January per **municipality** (one column per year) | SSB PxWebApi v2 |
| 5 | `frost_stations` | MET Frost API `/sources` | JSON | One weather **station** with its **municipality** (`municipalityId`) and county | API, one file |
| 6 | `frost_timeseries` | MET Frost API `/observations/availableTimeSeries` | JSON | One station that **has** daily mean temperature, with from/to dates | API, one file |
| 7 | `frost_observations` | MET Frost API `/observations` | JSON | Daily mean air temperature per **station × day** | API, one file per month per 100 stations |
| 8 | `seklima_daily_mean_temperature` | MET Seklima (manual export) | CSV `;` | Daily mean temperature per station × day | *Optional fallback* if Frost is down |

**How we get mean temperature per municipality** (the Bronze layer only collects the raw data; the calculation is Silver work – previewed in 4.5):
1. `frost_stations` tells which **municipality** each station is in.
2. `frost_timeseries` tells which stations actually **measure** `mean(air_temperature P1D)` – so we do not ask for data from rain-only or wind-only stations.
3. `frost_observations` gives the daily mean per station.
4. Silver: join 3 with 1 on station id, then **average all stations in a municipality per day**.

**Why it suits distributed processing:** dataset 1 alone is ~357 municipalities × 8 760 hours × ~5 groups ≈ 15 million rows
per year. Two–three years must be aggregated to days and joined with population, temperature and Norgespris data –
a typical Spark job (shuffle on `groupBy`, joins on municipality and date).

**Licences:** Elhub open data (NLOD), SSB (CC BY 4.0), MET Norway / Frost (CC BY 4.0).

**Reproducibility:** every file is kept unchanged in Bronze with source URL, time, size and SHA-256 checksum (section 3).
This matters extra for Elhub: the API only serves the last ~3 years, so **old months disappear from the source** –
our Bronze copy becomes the only copy. Never delete Bronze.

**Known quality risks** (checked with Spark in section 4): different municipality code formats, municipality `0` = "unknown",
broken Norwegian letters in some files, municipalities without a weather station, several daily-mean series per station
(different `timeOffset`), time zones (Elhub `+01:00/+02:00`, Frost UTC), revised Elhub data after month end, only yearly population.

---
# 3. Bronze layer (ingest and preserve)

**Design choices, and why:**

| Choice | Why |
|---|---|
| Keep files **as published** | Bronze is the proof of what the source delivered. All cleaning is done in Silver, where it can be tested and rerun. (Only gzip *transfer* compression is undone – see 3.6.) |
| Plain Python (`urllib`, `hashlib`, `csv`, `json`) for ingestion; Spark only for reading/checking | Downloading files is not a distributed task. Using only the standard library means nothing extra to install. Spark is used where it adds value (section 4 and Silver). |
| Credentials in **`.env`**, read at start | Keeps the Frost client ID out of the code and out of git (`.env` is in `.gitignore`). GitHub Actions gets the same value from a repository secret. |
| Folder layout `bronze/<source>/<dataset>/<period>/<run_id>__<file>` | Source and dataset visible in the path; `period` is the month (`2025-10`), `snapshot` or `manual`; `run_id` (UTC time of the run) shows *when* it was ingested and keeps old versions side by side. |
| One **manifest** file `bronze/_manifest.csv` | One line per file: run, source URL, file name, time, size, SHA-256, header line, schema status, status. Our traceability *and* monitoring log. |
| **Skip** a file whose checksum already exists for the dataset | Reruns are safe (idempotent) and never create duplicates. |
| Frost observations: **one request per month per 100 stations** | Keeps each answer small (≈ 3 000 values), well below Frost's limits, and gives the year/month folder structure from the definition. Months already stored are not requested again; the **2 newest months** are always re-requested because MET corrects recent values. |
| Compare the **header line** with the previous file of the dataset | A changed CSV header is flagged (`schema_status = changed`) but the file is kept, so Silver decides. (Not used for JSON.) |
| `required` flag per source | A failing **required** source makes the run fail (red in GitHub Actions). Optional sources only give a warning. |
| A `landing/` folder for manual files | Any dataset can be fed by hand if a download stops working. |
| Period in **`.env`** (dev = 2 months, prod = Nov 2023 → last complete month) | Small dev runs are quick; prod gets the full analysis period. Changing the period is a config change, not a code change. |

### 3.1 Imports
Only the Python standard library is needed for ingestion.

In [107]:
import base64                                  # build the HTTP Basic-auth header for the Frost API
import csv                                     # read and write the manifest file (handles commas/quotes in text)
import gzip                                    # data.elhub.no sends files gzip-compressed
import hashlib                                 # SHA-256 checksums to recognise identical files
import json                                    # read the Frost station list to know which stations to ask for
import logging                                 # log messages that also show up in GitHub Actions
import os                                      # environment variables and file paths
import shutil                                  # copy files and delete temp folders
import time                                    # measure run time and wait between download retries
import urllib.error                            # tells us the HTTP status code when a request fails (e.g. 404)
import urllib.request                          # download files over HTTPS (standard library, no pip install needed)
import zipfile                                 # Elhub's ".csv" exports are really ZIP files with a CSV inside
from datetime import datetime, timezone, date  # timestamps (always UTC) and month calculations

logging.basicConfig(level=logging.INFO,                                        # show INFO and above
                    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")  # time - logger - level - text
logger = logging.getLogger("bronze")                                          # named logger for this layer

### 3.2 Configuration: credentials, environment and folders

**Credentials:** `load_env_file()` reads `KEY=value` lines from `.env` (repo root) into environment variables.
It looks in the current folder and the parent folder, because the notebook runs from `notebooks/` and the script from the repo root.
Values that are already set (e.g. by GitHub Actions secrets) are **not** overwritten. We only print *which* keys were found – never the values.

**Environment:** `NORGESPRIS_ENV` and `NORGESPRIS_OUTPUT_DIR` are set by the workflows. When they are missing (you run the notebook yourself) we fall back to dev in your home folder.

**Period (which months to fetch from Frost)** is *configuration*, not code, so it lives in `.env`:

| Key in `.env` | Example | Meaning |
|---|---|---|
| `NORGESPRIS_DEV_START_MONTH` | `2025-09` | first month in dev |
| `NORGESPRIS_DEV_END_MONTH` | `2025-10` | last month in dev |
| `NORGESPRIS_PROD_START_MONTH` | `2023-11` | first month in prod |
| `NORGESPRIS_PROD_END_MONTH` | *(empty)* | last month in prod – empty = last complete month |

Why: the group can change the period without touching (and re-testing) the code, and dev and prod read the same code with different values.
`month_range()` reads the two keys that belong to the current environment and stops with a clear message if the start month is missing or not written as `YYYY-MM`.
It is called when the Frost requests are built (3.8), not when the code is loaded, so tests and the other sources still work without it.

In [108]:
def load_env_file(candidates=(".env", os.path.join("..", ".env"))):
    """Read KEY=value lines from the first .env file found into os.environ (does not overwrite existing values)."""
    for path in candidates:                                   # repo root seen from the root or from notebooks/
        if not os.path.exists(path):
            continue
        loaded = []
        with open(path, encoding="utf-8") as f:
            for line in f:
                line = line.strip()                           # remove spaces and the line break
                if not line or line.startswith("#") or "=" not in line:  # skip empty lines and comments
                    continue
                key, value = line.split("=", 1)               # split at the FIRST "=" only
                key = key.replace("export ", "").strip()      # allow "export KEY=value" too
                value = value.strip().strip('"').strip("'")   # remove spaces and surrounding quotes
                if key and key not in os.environ:             # real environment variables win (GitHub secrets)
                    os.environ[key] = value
                    loaded.append(key)
        logger.info(f"Loaded {loaded or 'no new keys'} from {os.path.abspath(path)}")  # names only, never values
        return
    logger.info("No .env file found - using environment variables only")

load_env_file()                                               # run it once when the code is loaded

ENV = os.environ.get("NORGESPRIS_ENV", "dev")                     # "dev" or "prod"; default dev when run by hand
DATA_ROOT = os.environ.get("NORGESPRIS_OUTPUT_DIR",               # root folder chosen by the workflow ...
                           os.path.expanduser(f"~/norgespris-data/{ENV}"))  # ... or ~/norgespris-data/dev locally

LANDING_DIR = os.path.join(DATA_ROOT, "landing")              # put manually exported files here
BRONZE_DIR = os.path.join(DATA_ROOT, "bronze")                # raw files, never edited
MANIFEST_PATH = os.path.join(BRONZE_DIR, "_manifest.csv")     # one line per file we have seen

REDOWNLOAD_RECENT_MONTHS = 2                                  # always re-fetch the newest months (MET corrects recent data)
FROST_STATIONS_PER_REQUEST = 100                              # stations per observations request (keeps answers small)


def month_range():
    """(start, end) months for the current environment from .env; end is None = up to the last complete month."""
    prefix = f"NORGESPRIS_{ENV.upper()}"                       # NORGESPRIS_DEV or NORGESPRIS_PROD
    start = os.environ.get(f"{prefix}_START_MONTH", "").strip()  # e.g. "2025-09"
    end = os.environ.get(f"{prefix}_END_MONTH", "").strip() or None  # empty -> None
    for key, value in ((f"{prefix}_START_MONTH", start), (f"{prefix}_END_MONTH", end)):
        if value is None and key.endswith("END_MONTH"):        # an empty end month is allowed
            continue
        if len(value) != 7 or value[4] != "-" or not (value[:4] + value[5:]).isdigit():  # must look like YYYY-MM
            raise ValueError(f"{key} must be set as YYYY-MM in .env (or as environment variable), got '{value}'")
    return start, end

2026-11-10 23:12:07,813 - bronze - INFO - Loaded no new keys from /Users/x/UiS/DAT535 Data Engineering/dat535-group24/.env


### 3.3 Source registry
One dictionary per dataset – **adding a dataset = adding one entry here.**

* `snapshot` – one file with everything, fetched every run (unchanged content is skipped by checksum).
  Elhub's `.csv` export accepts **no** `startDate`/`endDate` (HTTP 400), so it always returns the last ~3 years in one file.
* `frost_observations` – many requests: every month × every group of 100 stations (built in 3.8).
* `manual` – no download; only files placed in `landing/<source>/<dataset>/`.
* `sep` – column separator for CSV files (`None` = JSON).
* `required` – if `False`, a failed download is only a warning.
* `auth` – `"frost"` = send the Frost client ID.

**The three Frost calls** (the first two are the calls from our project description, the third is added by us):

| Dataset | URL | Why |
|---|---|---|
| `frost_stations` | `/sources/v0.jsonld?types=SensorSystem&validtime=2023-01-01/2027-01-01` | every station that existed in our period, **with its municipality** |
| `frost_timeseries` | `/observations/availableTimeSeries/v0.jsonld?elements=mean(air_temperature P1D)&referencetime=2023-11-01/2027-01-01` | only the stations that **have** daily mean temperature – about a third of all stations, so far fewer requests |
| `frost_observations` | `/observations/v0.jsonld?elements=mean(air_temperature P1D)&sources=SN…,SN…&referencetime=<month>` | the actual daily means |

The space in `mean(air_temperature P1D)` is written `%20` in a URL.

Every dataset **also** picks up files from its landing folder.

In [109]:
ELHUB_API = "https://api.elhub.no/energy-data/v0"             # Elhub Energy Data API (open, no login)
FROST_API = "https://frost.met.no"                            # MET Norway's Frost API (needs a client ID)
FROST_ELEMENT = "mean(air_temperature%20P1D)"                 # daily mean air temperature (%20 = space)

SSB_URL = ("https://data.ssb.no/api/pxwebapi/v2/tables/07459/data?lang=no"      # SSB table 07459, Norwegian labels
           "&valueCodes[Region]=*&codelist[Region]=agg_KommSummer"             # all municipalities, today's structure
           "&outputValues[Region]=aggregated"                                  # sums for merged municipalities
           "&valueCodes[ContentsCode]=*&valueCodes[Tid]=from(2023)"            # population, years 2023 ->
           "&outputFormat=csv&outputFormatParams=separatorsemicolon")          # CSV with ';' like the manual export
# Sex and age are left out of the query -> SSB sums them ("elimination"), so we get total population.

SOURCES = [                                                                    # the datasets the pipeline handles
    {"source": "elhub", "dataset": "consumption_per_group_municipality_hour",  # hourly kWh per municipality and group
     "mode": "snapshot", "sep": ";", "required": True,                         # .csv export = last ~3 years, no date filter
     "url": ELHUB_API + "/municipalities.csv?dataset=CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR"},
    {"source": "elhub", "dataset": "norgespris_consumption_per_group_eac_mba", # kWh with/without Norgespris per price area
     "mode": "snapshot", "sep": ",", "required": False,
     "url": ELHUB_API + "/price-areas.csv?dataset=NORGESPRIS_CONSUMPTION_PER_GROUP_EAC_MBA"},
    {"source": "elhub", "dataset": "norgespris_count_per_municipality_consumption_group",  # Norgespris meters per municipality
     "mode": "snapshot", "sep": ",", "required": False,
     "url": "https://data.elhub.no/download/norgespris_count_per_municipality_consumption_group/"
            "norgespris_count_per_municipality_consumption_group-all-en-0000-00-00.csv"},
    {"source": "ssb", "dataset": "population_07459",                          # population per municipality
     "mode": "snapshot", "sep": ";", "required": True, "url": SSB_URL},
    {"source": "met", "dataset": "frost_stations",                            # 1) station -> municipality
     "mode": "snapshot", "sep": None, "required": True, "auth": "frost", "ext": "json",
     "url": FROST_API + "/sources/v0.jsonld?types=SensorSystem&validtime=2023-01-01/2027-01-01"},
    {"source": "met", "dataset": "frost_timeseries",                          # 2) which stations have daily mean temp
     "mode": "snapshot", "sep": None, "required": True, "auth": "frost", "ext": "json",
     "url": FROST_API + "/observations/availableTimeSeries/v0.jsonld?elements=" + FROST_ELEMENT
                      + "&referencetime=2023-11-01/2027-01-01"},
    {"source": "met", "dataset": "frost_observations",                        # 3) daily mean temp per station
     "mode": "frost_observations", "sep": None, "required": True, "auth": "frost", "ext": "json",
     "url": FROST_API + "/observations/v0.jsonld?elements=" + FROST_ELEMENT
                      + "&sources={sources}&referencetime={start}/{end}"},
    {"source": "met", "dataset": "seklima_daily_mean_temperature",            # manual fallback, not needed with Frost
     "mode": "manual", "sep": ";", "required": False},
]

MANIFEST_COLUMNS = ["run_id", "env", "source", "dataset", "period", "origin",  # columns of _manifest.csv
                    "original_filename", "bronze_path", "ingested_at_utc", "size_bytes",
                    "sha256", "header", "schema_status", "status"]

### 3.4 Helper functions: checksum, header and manifest
* **SHA-256** is a fingerprint of the content: same content → same checksum. Reading in 1 MB blocks means a big file never has to fit in memory.
* **ZIP files:** Elhub's `.csv` export is a ZIP containing Spark-style `part-00000-….csv` files. A ZIP stores timestamps, so its bytes change on
  every download even when the data is the same. For ZIPs we therefore checksum the **files inside**, and read the header from the first CSV inside.
* The **header line** is stored in the manifest to detect schema changes in CSV files. For JSON (Frost) there is no header line, so we store `(json)`.
* The **manifest** is only appended to, never rewritten, so it holds the history of all runs.

In [110]:
def sha256_of(path):
    """Return the SHA-256 checksum of a file."""
    h = hashlib.sha256()                                       # empty checksum object
    with open(path, "rb") as f:                                # binary mode: we hash the exact bytes
        for block in iter(lambda: f.read(1024 * 1024), b""):   # read 1 MB at a time until end of file
            h.update(block)                                    # add the block to the checksum
    return h.hexdigest()                                       # 64-character hex string


def content_sha256(path):
    """Checksum of the data: for a ZIP the files inside (in name order), otherwise the file itself."""
    if not zipfile.is_zipfile(path):                           # normal file (CSV/JSON)
        return sha256_of(path)
    h = hashlib.sha256()
    with zipfile.ZipFile(path) as zf:                          # open the archive
        for name in sorted(zf.namelist()):                     # same order every time
            with zf.open(name) as member:                      # read one file inside the ZIP
                for block in iter(lambda: member.read(1024 * 1024), b""):  # 1 MB at a time
                    h.update(block)
    return h.hexdigest()


def read_header(path):
    """First line of a CSV (or of the first CSV inside a ZIP), without BOM and line break; '(json)' for JSON."""
    if path.lower().endswith(".json"):                         # JSON has no header line to compare
        return "(json)"
    if zipfile.is_zipfile(path):                               # ZIP -> look at the first CSV inside
        with zipfile.ZipFile(path) as zf:
            csvs = sorted(n for n in zf.namelist() if n.lower().endswith(".csv"))  # CSV members only
            if not csvs:                                       # a ZIP without CSV: nothing to compare
                return ""
            with zf.open(csvs[0]) as member:
                first = member.readline(2000)                  # first line of the first CSV
    else:
        with open(path, "rb") as f:                            # binary mode, so odd characters cannot crash us
            first = f.readline(2000)                           # first line only, max 2000 bytes
    return first.decode("utf-8-sig", errors="replace").strip() # utf-8-sig removes the BOM, strip removes \r\n


def read_manifest():
    """Return all rows of the manifest as a list of dicts (empty list on the first run)."""
    if not os.path.exists(MANIFEST_PATH):                      # first run: no manifest yet
        return []
    with open(MANIFEST_PATH, newline="", encoding="utf-8") as f:  # newline="" is required by the csv module
        return list(csv.DictReader(f))                         # one dict per row, keys = column names


def append_manifest(row):
    """Append one row to the manifest (writes the column names if the file is new)."""
    os.makedirs(BRONZE_DIR, exist_ok=True)                     # make sure the bronze folder exists
    is_new = not os.path.exists(MANIFEST_PATH)                 # do we need to write the column names?
    with open(MANIFEST_PATH, "a", newline="", encoding="utf-8") as f:  # "a" = append, never overwrite
        writer = csv.DictWriter(f, fieldnames=MANIFEST_COLUMNS, restval="")  # missing keys become ""
        if is_new:
            writer.writeheader()                               # column names first in a new manifest
        writer.writerow(row)                                   # this file's metadata

### 3.5 Month helpers
Frost observations are requested month by month. The newest month we ask for in prod is the **last complete month**.

In [111]:
def month_list(start, end):
    """List of 'YYYY-MM' strings from start to end, both included."""
    y, m = map(int, start.split("-"))                          # start year and month as numbers
    ey, em = map(int, end.split("-"))                          # end year and month as numbers
    months = []
    while (y, m) <= (ey, em):                                  # tuples compare year first, then month
        months.append(f"{y:04d}-{m:02d}")                      # e.g. "2025-09"
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)           # next month (December -> January next year)
    return months


def next_month(month):
    """'2025-12' -> '2026-01'."""
    y, m = map(int, month.split("-"))                          # split into numbers
    y, m = (y + 1, 1) if m == 12 else (y, m + 1)               # one month forward
    return f"{y:04d}-{m:02d}"


def last_complete_month(today=None):
    """The month before the current one, e.g. '2026-10' in November 2026."""
    today = today or date.today()                              # allow a fixed date in tests
    y, m = (today.year - 1, 12) if today.month == 1 else (today.year, today.month - 1)  # one month back
    return f"{y:04d}-{m:02d}"

### 3.6 Download: login, retries and gzip

* **Frost login:** HTTP *Basic authentication* with the **client ID as user name and an empty password** – that is what Frost's documentation asks for when reading open data.
  The client **secret** is only needed for OAuth2 (restricted data), so it is stored in `.env` but not sent.
* **Retries:** servers sometimes fail for a moment, so we try 3 times with a growing pause. But **not** for answers like 400/401/403/404 – those mean the request itself is wrong (or there is no data), and asking again will not help.
* **gzip:** `data.elhub.no` sends files gzip-compressed (the file starts with the bytes `1f 8b`). A browser unpacks this automatically, Python's `urllib` does not, so we unpack it. Gzip is only *packing for the transfer*; ZIP files are what Elhub actually publishes, so those are kept as they are (3.7).

In [112]:
def gunzip_if_needed(path):
    """If the file is gzip-compressed, replace it with the unpacked content."""
    with open(path, "rb") as f:
        if f.read(2) != b"\x1f\x8b":                           # gzip files always start with these two bytes
            return                                             # not gzip -> nothing to do
    with gzip.open(path, "rb") as src, open(path + ".unpacked", "wb") as dst:  # read compressed, write plain
        shutil.copyfileobj(src, dst)                           # stream, so big files are fine
    os.replace(path + ".unpacked", path)                       # swap the unpacked file in
    logger.info(f"  unpacked gzip: {os.path.basename(path)}")


def auth_headers(source_cfg):
    """Extra HTTP headers for sources that need a login. {} = open source, None = login missing."""
    if source_cfg.get("auth") != "frost":                      # Elhub and SSB are open
        return {}
    client_id = os.environ.get("FROST_CLIENT_ID", "").strip()  # from .env or a GitHub secret
    if not client_id or client_id.startswith("your-"):         # missing, or still the placeholder from .env
        return None
    token = base64.b64encode(f"{client_id}:".encode()).decode()  # "<client id>:" = empty password (Frost docs)
    return {"Authorization": f"Basic {token}"}


def download(url, target_path, headers=None, attempts=3):
    """Download url to target_path; retry server/network errors, raise the last error if all attempts fail."""
    all_headers = {"User-Agent": "DAT535-group24-student-project"}  # say politely who we are
    all_headers.update(headers or {})                          # add e.g. the Authorization header
    for attempt in range(1, attempts + 1):                     # attempt 1, 2, 3
        try:
            req = urllib.request.Request(url, headers=all_headers)                    # build the request
            with urllib.request.urlopen(req, timeout=300) as resp, open(target_path, "wb") as out:  # max 5 min
                shutil.copyfileobj(resp, out)                  # stream the response to the file
            gunzip_if_needed(target_path)                      # undo gzip transfer compression (data.elhub.no)
            return                                             # success -> stop retrying
        except urllib.error.HTTPError as e:                    # the server answered with an error code
            if e.code in (400, 401, 403, 404):                 # our request is wrong / no access / no data
                raise                                          # -> retrying will not help
            error = e                                          # 429, 500, 503 ... -> worth another try
        except Exception as e:                                 # network problem, timeout ...
            error = e
        logger.warning(f"Download attempt {attempt}/{attempts} failed for {url}: {error}")
        if attempt == attempts:                                # the last attempt also failed ...
            raise error                                        # ... let the caller record the failure
        time.sleep(5 * attempt)                                # wait 5 s, then 10 s

### 3.7 Ingest one file into Bronze
The heart of the Bronze layer. For each file:
1. compute checksum and read the header,
2. **duplicate?** same checksum already stored for this dataset → record `skipped_duplicate` and stop,
3. **schema check** (CSV only) against the newest earlier file of the dataset → `first_file`, `ok` or `changed`,
4. copy the file unchanged to `bronze/<source>/<dataset>/<period>/<run_id>__<filename>`;
   if it is a **ZIP**, it is saved with the `.zip` ending and *also* unpacked into `<run_id>__<name>_unzipped/` next to it
   (Spark cannot read ZIP files; the ZIP stays the untouched original),
5. write one manifest line.

`state` is a small dict with the run ID, all manifest rows so far and counters for this run.

In [113]:
def ingest_file(state, src_path, source, dataset, period, origin, original_filename):
    """Copy one file unchanged into Bronze unless an identical file is already there."""
    checksum = content_sha256(src_path)                         # fingerprint of the content (inside a ZIP: the files)
    header = read_header(src_path)                              # column-names line, or "(json)"
    earlier = [r for r in state["manifest"]                     # earlier files of the SAME dataset ...
               if r["dataset"] == dataset and r["status"] == "ingested"]  # ... that were actually stored

    row = {"run_id": state["run_id"], "env": ENV, "source": source, "dataset": dataset,  # metadata for the manifest
           "period": period, "origin": origin, "original_filename": original_filename,
           "ingested_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
           "size_bytes": os.path.getsize(src_path), "sha256": checksum, "header": header}

    if any(r["sha256"] == checksum for r in earlier):           # exact same content seen before?
        row["bronze_path"] = next(r["bronze_path"] for r in reversed(earlier) if r["sha256"] == checksum)
        row["status"] = "skipped_duplicate"                     # yes -> do not store it again
        state["counts"]["skipped_duplicate"] += 1
        logger.info(f"  skip (already ingested): {dataset} {period} {original_filename}")
    else:
        if not earlier:                                         # first file ever for this dataset
            row["schema_status"] = "first_file"
        elif earlier[-1]["header"] == header:                   # same columns as the newest earlier file
            row["schema_status"] = "ok"
        else:                                                   # columns changed -> keep the file, but flag it
            row["schema_status"] = "changed"
            logger.warning(f"  HEADER CHANGED for {dataset}: check before Silver!")
        is_zip = zipfile.is_zipfile(src_path)                                 # Elhub exports are ZIPs
        stored_name = original_filename
        if is_zip and not stored_name.lower().endswith(".zip"):               # give it the correct file ending
            stored_name = os.path.splitext(stored_name)[0] + ".zip"
        target_dir = os.path.join(BRONZE_DIR, source, dataset, period)        # e.g. bronze/met/frost_observations/2025-10
        os.makedirs(target_dir, exist_ok=True)                                # create the folder if missing
        target = os.path.join(target_dir, f"{state['run_id']}__{stored_name}")  # run_id keeps versions apart
        shutil.copy2(src_path, target)                                        # byte-for-byte copy of the original
        if is_zip:                                                            # unpack a copy for Spark
            with zipfile.ZipFile(target) as zf:
                zf.extractall(target[:-len(".zip")] + "_unzipped")            # <run_id>__<name>_unzipped/part-....csv
        row["bronze_path"] = os.path.relpath(target, BRONZE_DIR)              # path relative to bronze/
        row["status"] = "ingested"
        state["counts"]["ingested"] += 1
        logger.info(f"  ingested: {row['bronze_path']}")

    append_manifest(row)                                        # write the line to disk right away
    state["manifest"].append(row)                               # and remember it for the rest of the run

### 3.8 Download → ingest, and the Frost observation requests

**`download_and_ingest`** downloads to a temporary file first, so only complete files are ingested. Every outcome ends up in the manifest:

| Outcome | Manifest status | Effect on the run |
|---|---|---|
| file stored / identical file already there | `ingested` / `skipped_duplicate` | – |
| Frost answers **404** for a month and group of stations | `no_data` | none – Frost says "404" when the stations have no values in that period |
| login missing (no `FROST_CLIENT_ID`) | `failed: …` / `warning: …` | required source → run **fails**, with a clear message |
| any other error | `failed: …` / `warning: …` | required source → run **fails** |

**`frost_observation_requests`** builds the list of observation requests:
1. read the newest `frost_timeseries` file from Bronze → the station ids that have daily mean temperature (`SN18700:0` → `SN18700`),
2. sort them and cut them into groups of 100,
3. for every month in the period and every group: one URL and a file name like `frost_obs_2025-10_g03.json`,
4. leave out requests whose file we already have – except for the 2 newest months.

In [114]:
def download_and_ingest(state, cfg, url, period, filename):
    """Download url to a temp file and ingest it; record every outcome in the manifest."""
    source, dataset = cfg["source"], cfg["dataset"]
    level = "failed" if cfg["required"] else "warning"           # how bad a failure is for this source
    base_row = {"run_id": state["run_id"], "env": ENV, "source": source, "dataset": dataset,
                "period": period, "origin": url, "original_filename": filename,
                "ingested_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    headers = auth_headers(cfg)                                  # {} for open sources, None if login is missing
    if headers is None:                                          # Frost without a client ID
        state["counts"][level] += 1
        append_manifest(dict(base_row, status=f"{level}: FROST_CLIENT_ID missing (set it in .env)"))
        logger.error(f"  {level.upper()}: {dataset} needs FROST_CLIENT_ID (in .env or as GitHub secret)")
        return
    tmp_path = os.path.join(state["tmp_dir"], filename)          # temp file in bronze/_tmp/<run_id>/
    try:
        download(url, tmp_path, headers)                         # may raise after failed attempts
        ingest_file(state, tmp_path, source, dataset, period, url, filename)  # checks + copy + manifest
    except urllib.error.HTTPError as e:
        if e.code == 404 and cfg["mode"] == "frost_observations":  # Frost: "no data for these stations/month"
            state["counts"]["no_data"] += 1
            row = dict(base_row, status="no_data")
            append_manifest(row)
            state["manifest"].append(row)                        # remember: do not ask again (unless recent)
            logger.info(f"  no data: {filename}")
        else:
            state["counts"][level] += 1
            append_manifest(dict(base_row, status=f"{level}: HTTP {e.code} {e.reason}"[:200]))
            logger.error(f"  {level.upper()}: {url} (HTTP {e.code} {e.reason})")
    except Exception as e:                                       # network error etc.
        state["counts"][level] += 1
        append_manifest(dict(base_row, status=f"{level}: {type(e).__name__} {e}"[:200]))
        logger.error(f"  {level.upper()}: {url} ({e})")
    finally:
        if os.path.exists(tmp_path):                             # always clean up the temp file
            os.remove(tmp_path)


def frost_station_ids(state):
    """Station ids that have daily mean temperature, from the newest frost_timeseries file in Bronze."""
    rows = [r for r in state["manifest"]                         # stored availableTimeSeries files
            if r["dataset"] == "frost_timeseries"
            and r["status"] in ("ingested", "skipped_duplicate") and r.get("bronze_path")]
    if not rows:                                                 # step 2 did not succeed (yet)
        return []
    path = os.path.join(BRONZE_DIR, rows[-1]["bronze_path"])     # newest version
    with open(path, encoding="utf-8") as f:
        data = json.load(f).get("data", [])                      # Frost puts the records in "data"
    return sorted({d["sourceId"].split(":")[0] for d in data})   # "SN18700:0" -> "SN18700", no duplicates


def frost_observation_requests(state, cfg):
    """List of (url, month, filename): every month x every group of stations, minus what we already have."""
    ids = frost_station_ids(state)                               # stations to ask for
    if not ids:
        logger.error("  no station list from frost_timeseries - cannot request observations")
        return []
    n = FROST_STATIONS_PER_REQUEST
    groups = [ids[i:i + n] for i in range(0, len(ids), n)]       # [[SN..100 ids], [SN..], ...]
    start, end = month_range()                                   # period for dev or prod, from .env
    months = month_list(start, end or last_complete_month())     # None = up to the last complete month
    recent = set(months[-REDOWNLOAD_RECENT_MONTHS:])             # always re-request the newest months
    have = {r["original_filename"] for r in state["manifest"]    # files we already got (or know are empty)
            if r["dataset"] == cfg["dataset"] and r["status"] in ("ingested", "skipped_duplicate", "no_data")}
    requests = []
    for month in months:
        for g, group in enumerate(groups, start=1):
            filename = f"frost_obs_{month}_g{g:02d}.json"         # e.g. frost_obs_2025-10_g03.json
            if filename in have and month not in recent:         # already stored and not recent -> skip
                continue
            url = cfg["url"].format(sources=",".join(group),     # SN18700,SN18950,...
                                    start=f"{month}-01",          # first day of the month
                                    end=f"{next_month(month)}-01")  # first day of next month (end not included)
            requests.append((url, month, filename))
    logger.info(f"  {len(ids)} stations in {len(groups)} groups -> {len(requests)} requests")
    return requests

### 3.9 The whole Bronze run
Loops over the source registry **in order** – the order matters: `frost_timeseries` must be stored before `frost_observations` can use it.
Returns `status` and `elapsed_time`; `run_pipeline.py` turns `failed` into exit code 1 (red run in GitHub Actions).

In [115]:
def run_bronze():
    """Run Bronze ingestion for all sources and return a summary dict."""
    start_time = time.time()
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")  # e.g. 20261109T201500Z
    state = {"run_id": run_id,                                 # shared state for the helper functions
             "tmp_dir": os.path.join(BRONZE_DIR, "_tmp", run_id),
             "manifest": read_manifest(),                      # everything ingested in earlier runs
             "counts": {"ingested": 0, "skipped_duplicate": 0, "no_data": 0, "warning": 0, "failed": 0}}
    os.makedirs(state["tmp_dir"], exist_ok=True)
    logger.info(f"Bronze run {run_id} (env={ENV}) -> {BRONZE_DIR}")

    for cfg in SOURCES:                                        # one dataset at a time, in registry order
        source, dataset, mode = cfg["source"], cfg["dataset"], cfg["mode"]
        logger.info(f"[{source}/{dataset}] mode={mode}")
        ext = cfg.get("ext", "csv")                            # file extension (Frost gives JSON)

        if mode == "snapshot":                                 # one file with everything
            download_and_ingest(state, cfg, cfg["url"], "snapshot", f"{dataset}.{ext}")
        elif mode == "frost_observations":                     # many small requests
            if auth_headers(cfg) is None:                      # no login -> one clear failure, not hundreds
                download_and_ingest(state, cfg, cfg["url"], "-", "frost_obs.json")
            else:
                for url, month, filename in frost_observation_requests(state, cfg):
                    download_and_ingest(state, cfg, url, month, filename)

        landing = os.path.join(LANDING_DIR, source, dataset)   # manual files for this dataset
        os.makedirs(landing, exist_ok=True)                    # create it so you can see where files go
        for name in sorted(os.listdir(landing)):               # every file in the landing folder
            if name.lower().endswith((".csv", ".json", ".zip")):  # only data files
                ingest_file(state, os.path.join(landing, name), source, dataset,
                            "manual", f"landing/{source}/{dataset}", name)  # duplicates skipped by checksum

    shutil.rmtree(state["tmp_dir"], ignore_errors=True)        # remove the temp folder
    elapsed = time.time() - start_time
    logger.info(f"Bronze done in {elapsed:.1f}s: {state['counts']}")
    status = "success" if state["counts"]["failed"] == 0 else "failed"  # a failed REQUIRED source -> failed run
    return {"status": status, "elapsed_time": elapsed, **state["counts"]}

### 3.10 Run it

**Before the first run:**
1. Open the file `.env` in the repo root and replace `your-frost-client-id` with your Frost **client ID** (and the secret, for completeness).
   No `.env` yet? Copy `.env.example` to `.env`. The file is in `.gitignore`, so it is never pushed to GitHub.
2. Check the four `NORGESPRIS_…_MONTH` lines in `.env` (the period to fetch).
3. Re-run cell 3.2 after editing `.env` (or restart the kernel), so the new values are loaded.

The next cell prints the landing folders (only needed for manual files) and whether the Frost login was found.

Running the run cell twice is safe: unchanged files are skipped as `skipped_duplicate`, and only the 2 newest months of Frost data are requested again.
The first prod run makes a few hundred small Frost requests and downloads 3 years of Elhub data – expect it to take a while.

In [116]:
for cfg in SOURCES:                                            # show where each dataset's manual files go
    path = os.path.join(LANDING_DIR, cfg["source"], cfg["dataset"])  # landing/<source>/<dataset>
    os.makedirs(path, exist_ok=True)                           # create the folder now
    print(f"{cfg['mode']:18s} {path}")                         # mode + folder
print("\nFrost login found:", auth_headers({"auth": "frost"}) is not None)  # True/False only, never the value
print(f"Period for {ENV}:", month_range())                    # (start, end) from .env; None = last complete month

snapshot           /Users/x/norgespris-data/dev/landing/elhub/consumption_per_group_municipality_hour
snapshot           /Users/x/norgespris-data/dev/landing/elhub/norgespris_consumption_per_group_eac_mba
snapshot           /Users/x/norgespris-data/dev/landing/elhub/norgespris_count_per_municipality_consumption_group
snapshot           /Users/x/norgespris-data/dev/landing/ssb/population_07459
snapshot           /Users/x/norgespris-data/dev/landing/met/frost_stations
snapshot           /Users/x/norgespris-data/dev/landing/met/frost_timeseries
frost_observations /Users/x/norgespris-data/dev/landing/met/frost_observations
manual             /Users/x/norgespris-data/dev/landing/met/seklima_daily_mean_temperature

Frost login found: True
Period for dev: ('2026-10', '2026-10')


In [117]:
result = run_bronze()                                          # run the whole Bronze layer
result                                                         # show the summary

2026-11-10 23:12:08,033 - bronze - INFO - Bronze run 20261110T221208Z (env=dev) -> /Users/x/norgespris-data/dev/bronze
2026-11-10 23:12:08,033 - bronze - INFO - [elhub/consumption_per_group_municipality_hour] mode=snapshot
2026-11-10 23:12:24,847 - bronze - INFO -   skip (already ingested): consumption_per_group_municipality_hour snapshot consumption_per_group_municipality_hour.csv
2026-11-10 23:12:24,865 - bronze - INFO - [elhub/norgespris_consumption_per_group_eac_mba] mode=snapshot
2026-11-10 23:12:25,438 - bronze - INFO -   skip (already ingested): norgespris_consumption_per_group_eac_mba snapshot norgespris_consumption_per_group_eac_mba.csv
2026-11-10 23:12:25,439 - bronze - INFO - [elhub/norgespris_count_per_municipality_consumption_group] mode=snapshot
2026-11-10 23:12:25,928 - bronze - INFO -   unpacked gzip: norgespris_count_per_municipality_consumption_group.csv
2026-11-10 23:12:25,943 - bronze - INFO -   skip (already ingested): norgespris_count_per_municipality_consumption_

{'status': 'success',
 'elapsed_time': 35.537827253341675,
 'ingested': 14,
 'skipped_duplicate': 4,
 'no_data': 0,
 'warning': 0,
 'failed': 0}

### 3.11 The manifest (traceability and monitoring)
Evidence for section 3 of the definition: where each file came from, when, its checksum, and whether it was new, a duplicate, empty, a schema change or a failure. The manifest is small, so pandas is enough here.

In [118]:
import pandas as pd                                            # small table -> pandas is fine

manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna("")   # read everything as text
cols = ["run_id", "dataset", "period", "original_filename", "status", "schema_status", "size_bytes"]
display(manifest[cols].tail(20))                               # newest 20 lines
print(manifest.groupby(["dataset", "status"]).size())          # files per dataset and status (all runs)

,run_id,dataset,period,original_filename,status,schema_status,size_bytes
547,20261110T220621Z,frost_observations,2026-10,frost_obs_2026-10_g11.json,ingested,ok,2174203
548,20261110T220621Z,frost_observations,2026-10,frost_obs_2026-10_g12.json,ingested,ok,271790
549,20261110T221208Z,consumption_per_group_municipality_hour,snapshot,consumption_per_group_municipality_hour.csv,skipped_duplicate,,448468580
550,20261110T221208Z,norgespris_consumption_per_group_eac_mba,snapshot,norgespris_consumption_per_group_eac_mba.csv,skipped_duplicate,,1320866
551,20261110T221208Z,norgespris_count_per_municipality_consumption_...,snapshot,norgespris_count_per_municipality_consumption_...,skipped_duplicate,,15434175
552,20261110T221208Z,population_07459,snapshot,population_07459.csv,skipped_duplicate,,11224
553,20261110T221208Z,frost_stations,snapshot,frost_stations.json,ingested,ok,1337389
554,20261110T221208Z,frost_timeseries,snapshot,frost_timeseries.json,ingested,ok,1438863
555,20261110T221208Z,frost_observations,2026-10,frost_obs_2026-10_g01.json,ingested,ok,1875816
556,20261110T221208Z,frost_observations,2026-10,frost_obs_2026-10_g02.json,ingested,ok,1886312


dataset                                              status                                                                                 
consumption_per_group_municipality_hour              failed: HTTPError HTTP Error 400: Bad Request                                                2
                                                     ingested                                                                                     3
                                                     skipped_duplicate                                                                           13
frost_observations                                   failed: URLError <urlopen error [Errno 8] nodename nor servname provided, or not known>     15
                                                     ingested                                                                                   465
frost_stations                                       failed: URLError <urlopen error [Errno 8] nodename nor servname pr

---
# 4. Check Bronze with Spark and profile the data (evidence for section 2)

This proves the Bronze files can be read by Spark and measures what Silver will have to fix.
CSV columns are read **as text**: Bronze/profiling must not guess types – type conversion is a Silver decision.

### 4.1 SparkSession
Small, explicit settings: adaptive query execution on, 8 shuffle partitions (the default of 200 is far too many for this data size), 2 GB driver memory.

In [119]:
from pyspark.sql import SparkSession                            # entry point to Spark
from pyspark.sql import functions as F                          # column functions (count, min, max, ...)

spark = (SparkSession.builder
         .appName("DAT535-Group24-Bronze-Check")                # name in the Spark UI
         .config("spark.sql.adaptive.enabled", "true")          # let Spark tune partitions/joins at runtime
         .config("spark.sql.shuffle.partitions", "8")           # small data -> few shuffle partitions
         .config("spark.driver.memory", "2g")                   # enough for profiling
         .getOrCreate())                                        # reuse a running session if there is one
spark.sparkContext.setLogLevel("WARN")                          # hide Spark's INFO noise
print(spark.version)

4.3.0


### 4.2 Read the newest Bronze version of each dataset
* `newest_files` picks the newest version of **each file** (per period and file name) – a month of Frost data consists of several files (one per group of stations), and we need all of them.
* For a ZIP we read its unpacked folder (`…_unzipped/`); Spark reads every CSV in a folder and skips files starting with `_` or `.`.
* The separator is taken from the stored header line (more `;` than `,` → `;`), because the CSV inside Elhub's ZIP may differ from the files we downloaded by hand.
* We only remove the BOM character from the *column names* so we can refer to them – the data itself is not touched.
* JSON datasets (Frost) are read separately in 4.5.

In [120]:
def newest_files(dataset):
    """Absolute paths of the newest ingested version of every file of a dataset."""
    rows = manifest[(manifest.dataset == dataset) & manifest.status.isin(("ingested", "skipped_duplicate"))
                    & manifest.bronze_path.notna()]  # latest successful fetches with a stored path
    newest = rows.sort_values("run_id").groupby(["period", "original_filename"]).tail(1)  # newest per file
    paths = [os.path.join(BRONZE_DIR, p) for p in newest.bronze_path]                 # full paths
    return [p[:-len(".zip")] + "_unzipped" if p.endswith(".zip") else p for p in paths]  # ZIP -> its unpacked folder

def guess_sep(dataset, default):
    """Separator from the header line stored in the manifest: ';' if it has more ';' than ','."""
    rows = manifest[(manifest.dataset == dataset) & manifest.status.isin(("ingested", "skipped_duplicate"))]
    if rows.empty:                                              # nothing stored -> use the registry value
        return default
    header = rows.sort_values("run_id").header.iloc[-1]         # header of the newest file
    return ";" if header.count(";") > header.count(",") else ","

def read_bronze(cfg):
    """Read a Bronze CSV dataset with Spark, all columns as text; None if nothing to read."""
    files = newest_files(cfg["dataset"])                        # which files/folders to read
    if not files or cfg["sep"] is None:                         # nothing ingested, or JSON (Frost -> 4.5)
        return None
    sep = guess_sep(cfg["dataset"], cfg["sep"])                 # Elhub's ZIP may use another separator than our samples
    df = (spark.read
          .option("header", True)                               # first line = column names
          .option("inferSchema", False)                         # keep everything as text in Bronze
          .option("sep", sep)                                   # separator for this dataset
          .option("encoding", "UTF-8")
          .csv(files))                                          # list of files -> one DataFrame
    return df.toDF(*[c.replace("﻿", "").strip() for c in df.columns])  # remove BOM from column names only

bronze = {cfg["dataset"]: read_bronze(cfg) for cfg in SOURCES}  # one DataFrame per CSV dataset (None for JSON)

### 4.3 Row counts and columns
`count()` is an *action*: this is where Spark actually reads the files. These numbers go into the report as the Bronze row counts that Silver results are compared with.

In [121]:
for name, df in bronze.items():                                 # every CSV dataset
    if df is None:                                              # skip what is not ingested / JSON
        continue
    print(f"\n=== {name}: {df.count():,} rows, {len(df.columns)} columns")  # action -> Spark reads the files
    print(df.columns)                                           # column names
    df.show(3, truncate=False)                                  # first 3 rows


=== consumption_per_group_municipality_hour: 28,250,496 rows, 8 columns
['START_TIME', 'END_TIME', 'MUNICIPALITY_NUMBER', 'NAME', 'CONSUMPTION_GROUP', 'QUANTITY_KWH', 'METERING_POINT_COUNT', 'LAST_UPDATE_TIME']
+-----------------------------+-----------------------------+-------------------+------------------+-----------------+------------+--------------------+-----------------------------+
|START_TIME                   |END_TIME                     |MUNICIPALITY_NUMBER|NAME              |CONSUMPTION_GROUP|QUANTITY_KWH|METERING_POINT_COUNT|LAST_UPDATE_TIME             |
+-----------------------------+-----------------------------+-------------------+------------------+-----------------+------------+--------------------+-----------------------------+
|2026-10-10T13:00:00.000+02:00|2026-10-10T14:00:00.000+02:00|0000               |Ukjent tilhørighet|business         |1023.589000 |175                 |2026-10-24T17:38:55.680+02:00|
|2026-10-10T14:00:00.000+02:00|2026-10-10T15:00:00.000+0

### 4.4 Source-specific checks that Silver must handle
One small query per known risk. Each printed number is a finding for the report.

In [122]:
cons = bronze["consumption_per_group_municipality_hour"]        # Elhub hourly consumption
if cons is not None:
    cons.agg(F.min("START_TIME").alias("first_hour"),           # period covered by the files
             F.max("START_TIME").alias("last_hour"),
             F.countDistinct("MUNICIPALITY_NUMBER").alias("municipalities")).show(truncate=False)
    cons.groupBy("CONSUMPTION_GROUP").count().show()            # rows per consumption group
    print("Rows with municipality 0 ('unknown'):",
          cons.filter(F.col("MUNICIPALITY_NUMBER") == "0").count())       # cannot be joined to SSB -> handle in Silver
    print("Names with broken letters (√ instead of æøå):",
          cons.filter(F.col("NAME").contains("√")).select("NAME").distinct().count())  # encoding check
    print("Municipality numbers shorter than 4 digits:",
          cons.filter(F.length("MUNICIPALITY_NUMBER") < 4).select("MUNICIPALITY_NUMBER").distinct().count())

+-----------------------------+-----------------------------+--------------+
|first_hour                   |last_hour                    |municipalities|
+-----------------------------+-----------------------------+--------------+
|2023-11-10T00:00:00.000+01:00|2026-11-09T23:00:00.000+01:00|358           |
+-----------------------------+-----------------------------+--------------+



+-----------------+-------+
|CONSUMPTION_GROUP|  count|
+-----------------+-------+
|          private|9416832|
|         industry|9416832|
|         business|9416832|
+-----------------+-------+



Rows with municipality 0 ('unknown'): 0


Names with broken letters (√ instead of æøå): 0


Municipality numbers shorter than 4 digits: 0


In [123]:
pop = bronze["population_07459"]                                # SSB population
if pop is not None:
    region = pop.columns[0]                                     # region column ("Kommune"/"region")
    pop.select(region).show(3, truncate=False)                  # codes look like "K-3101 Halden"
    print("Rows that are not real municipalities:",
          pop.filter(F.col(region).rlike("Rest|Kontinentalsokkelen|[Uu]oppgitt")).count())  # remove in Silver

+------+
|Region|
+------+
|K-3101|
|K-3103|
|K-3105|
+------+
only showing top 3 rows
Rows that are not real municipalities: 1


### 4.5 Frost: stations, observations – and a preview of mean temperature per municipality

Frost answers in JSON with the records inside a list called `data`. Spark reads JSON directly
(`multiLine=True` because each file is one big JSON document, not one record per line), and `explode` turns the list into rows.

* **Stations:** one row per station with `municipalityId` (a number like `301` → we write it as `0301` with `lpad`, the same 4-digit form Elhub uses).
* **Observations:** each record is one station and one day, with a list of observations inside – usually one, but a station can have **several daily-mean series** with different `timeOffset` (when the 24-hour period starts, e.g. `PT0H` = midnight UTC, `PT6H` = 06 UTC). We count them first.
* **Preview (Silver logic, not saved):** keep one series per station and day (`PT0H` if it exists, otherwise the first one), join with the stations, and average per municipality and day.
  This shows that the data supports our question; the real, tested version comes in the Silver notebook.

In [124]:
def read_frost(dataset):
    """Read all newest Frost JSON files of a dataset with Spark and explode the 'data' list into rows."""
    files = newest_files(dataset)                               # all JSON files of the dataset
    if not files:
        print(f"-- {dataset}: nothing in Bronze yet (is FROST_CLIENT_ID set?)")
        return None
    raw = spark.read.option("multiLine", True).json(files)      # one JSON document per file
    return raw.select(F.explode("data").alias("d")).select("d.*")  # one row per element of "data"

stations = read_frost("frost_stations")                         # step 1: station -> municipality
observations = read_frost("frost_observations")                 # step 3: daily means

if stations is not None:
    stations = stations.select(
        F.col("id").alias("station_id"),                        # e.g. SN18700
        F.col("name").alias("station_name"),
        F.lpad(F.col("municipalityId").cast("string"), 4, "0").alias("municipality_no"),  # 301 -> "0301"
        F.col("municipality").alias("municipality_name"),
        F.col("county").alias("county_name"))
    print("Stations:", stations.count(),                       # all stations in the period
          "| with a municipality:", stations.filter(F.col("municipality_no").isNotNull()).count())
    stations.show(3, truncate=False)

Stations: 2304 | with a municipality: 1908
+----------+-----------------+---------------+-----------------+-----------+
|station_id|station_name     |municipality_no|municipality_name|county_name|
+----------+-----------------+---------------+-----------------+-----------+
|SN47230   |ÅKRA UNGDOMSSKOLE|1149           |KARMØY           |ROGALAND   |
|SN20952   |STATFJORD C      |2600           |NORDSJØEN        |NORDSJØEN  |
|SN23670   |E16 RYFOSS       |3454           |VANG             |INNLANDET  |
+----------+-----------------+---------------+-----------------+-----------+
only showing top 3 rows


In [125]:
if observations is not None:
    obs = (observations
           .select(F.split("sourceId", ":")[0].alias("station_id"),             # "SN18700:0" -> "SN18700"
                   F.to_date(F.substring("referenceTime", 1, 10)).alias("date"),  # "2025-10-01T00:00:00.000Z" -> date
                   F.explode("observations").alias("o"))                         # one row per observation
           .select("station_id", "date",
                   F.col("o.value").cast("double").alias("mean_temp_c"),         # the temperature in °C
                   F.col("o.timeOffset").alias("time_offset")))                 # which daily-mean series
    print("Observation rows:", obs.count())
    obs.groupBy("time_offset").count().orderBy("time_offset").show()             # how many series per offset
    obs.agg(F.min("date").alias("first_day"), F.max("date").alias("last_day"),
            F.countDistinct("station_id").alias("stations")).show()

Observation rows: 1489810


+-----------+------+
|time_offset| count|
+-----------+------+
|       PT0H|910343|
|       PT6H|579467|
+-----------+------+



+----------+----------+--------+
| first_day|  last_day|stations|
+----------+----------+--------+
|2024-01-01|2026-11-10|    1110|
+----------+----------+--------+



In [126]:
if stations is not None and observations is not None:
    from pyspark.sql import Window                               # to pick one series per station and day

    pick = Window.partitionBy("station_id", "date").orderBy(    # within one station and day ...
        F.when(F.col("time_offset") == "PT0H", 0).otherwise(1),  # ... prefer the PT0H series
        "time_offset")                                          # ... otherwise the first offset
    one_per_day = (obs.withColumn("rank", F.row_number().over(pick))
                      .filter("rank = 1").drop("rank"))         # exactly one value per station and day

    muni_temp = (one_per_day
                 .join(F.broadcast(stations), "station_id")     # small station table -> broadcast join (no shuffle of obs)
                 .filter(F.col("municipality_no").isNotNull())  # stations outside a municipality (sea, abroad)
                 .groupBy("municipality_no", "municipality_name", "date")
                 .agg(F.round(F.avg("mean_temp_c"), 1).alias("mean_temp_c"),     # average of all stations
                      F.count("*").alias("n_stations")))                        # how many stations it is based on

    muni_temp.orderBy("municipality_no", "date").show(10, truncate=False)
    print("Municipalities with at least one station:", muni_temp.select("municipality_no").distinct().count(),
          "of ~357 -> the rest need a fallback in Silver (county average or nearest station)")

+---------------+-----------------+----------+-----------+----------+
|municipality_no|municipality_name|date      |mean_temp_c|n_stations|
+---------------+-----------------+----------+-----------+----------+
|0301           |OSLO             |2024-01-01|-5.4       |12        |
|0301           |OSLO             |2024-01-02|-8.0       |12        |
|0301           |OSLO             |2024-01-03|-9.2       |12        |
|0301           |OSLO             |2024-01-04|-11.4      |12        |
|0301           |OSLO             |2024-01-05|-17.9      |12        |
|0301           |OSLO             |2024-01-06|-18.7      |12        |
|0301           |OSLO             |2024-01-07|-16.8      |12        |
|0301           |OSLO             |2024-01-08|-14.7      |12        |
|0301           |OSLO             |2024-01-09|-8.5       |12        |
|0301           |OSLO             |2024-01-10|-6.1       |11        |
+---------------+-----------------+----------+-----------+----------+
only showing top 10 

Municipalities with at least one station: 321 of ~357 -> the rest need a fallback in Silver (county average or nearest station)


---
# 6. How to run (summary)

| Where | Command |
|---|---|
| Notebook (dev) | Fill in `.env`, run all cells; data in `~/norgespris-data/dev/` |
| Terminal | `bash scripts/setup_env.sh` once, then `source ~/spark-env/bin/activate && python run_pipeline.py bronze` |
| Tests | `pytest -q tests/` |
| GitHub Actions – dev | push to `dev` (or run by hand) → `dat535-dev.yml`: tests, then the pipeline into `~/norgespris-data/dev` |
| GitHub Actions – prod | merge to `main` (+ approval) or run by hand → `dat535-prod.yml` |
| GitHub Actions – schedule | 5th of every month → `pipeline.yml` refreshes prod |

GitHub Actions does not see your `.env` (it is not in git):
* the client ID goes in as repository **secret** `FROST_CLIENT_ID`,
* the period goes in as repository **variables** `NORGESPRIS_DEV_START_MONTH`, … (*Settings → Secrets and variables → Actions → Variables*).
  If a variable is not set, the workflow file uses the same default values as `.env.example`.

The machine needs internet access to `api.elhub.no`, `data.elhub.no`, `data.ssb.no` and `frost.met.no`.

**Next notebook (02):** Silver – read the newest Bronze files with explicit schemas, clean, standardise and write Parquet.